# Target Transformation, Target Engineering and Feature Engineering

Some feature engineering was performed during the ETL.  They include:
- Noise calculation using traffic volumen & traffic mix (**noise**)
- Station Assignment by Haversine distance to the nearest noise detection station (**assigned_station**)
- Calculated distance of observation coordiates from I95 using Haversine distance between observation point to the nearest I95 line (**i95_distance**)

This notebook covers additional data pre-processing steps taken post EDA to enhance model performances.
- Creation of 'observation_rate' by tranforming 'OBSERVATION COUNT' with 'DURATION MINUNTES' and 'OBSERVER COUNT';
- Creation of new 'noiss-change' feature with historical average; and,
- Addressing extreme target imbalance with BoxCox transformation.

In [ ]:
# required imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer

/Users/sooneui/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [11]:
data = pd.read_parquet("/Users/sooneui/Documents/UCB_MIDS/Data/Capstone_Data/stations.parquet")
print("shape:", data.shape)


shape: (952074, 14)


In [12]:
data.columns

Index(['TAXON CONCEPT ID', 'COMMON NAME', 'SCIENTIFIC NAME',
       'OBSERVATION COUNT', 'STATE CODE', 'COUNTY', 'LOCALITY',
       'LOCALITY TYPE', 'LATITUDE', 'LONGITUDE', 'OBSERVATION DATE',
       'TIME OBSERVATIONS STARTED', 'NUMBER OBSERVERS', 'DURATION MINUTES'],
      dtype='object')

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 952074 entries, 0 to 952073
Data columns (total 14 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   TAXON CONCEPT ID           952074 non-null  object 
 1   COMMON NAME                952074 non-null  object 
 2   SCIENTIFIC NAME            952074 non-null  object 
 3   OBSERVATION COUNT          952074 non-null  object 
 4   STATE CODE                 952074 non-null  object 
 5   COUNTY                     952074 non-null  object 
 6   LOCALITY                   952074 non-null  object 
 7   LOCALITY TYPE              952074 non-null  object 
 8   LATITUDE                   952074 non-null  float64
 9   LONGITUDE                  952074 non-null  float64
 10  OBSERVATION DATE           952074 non-null  object 
 11  TIME OBSERVATIONS STARTED  948753 non-null  object 
 12  NUMBER OBSERVERS           945329 non-null  float64
 13  DURATION MINUTES           88

### 1. Observation_Rate - an alternate target

Convert target ('OBSERVATION COUNT') to numeric and dropping non numeric rows

In [14]:
print("Before cleaning:", data.shape)
# Cleaning up 41K nonnumeric data ("X") from our target
data['OBSERVATION COUNT'] = pd.to_numeric(data['OBSERVATION COUNT'], errors='coerce')
# Dropping non-numeric 'OBSERVATION COUNT' rows & null 'TIME OBSERVATIONS STARTED'
clean_df = data.dropna(subset=['OBSERVATION COUNT', 'TIME OBSERVATIONS STARTED'])
print("After cleaning:", clean_df.shape)

Before cleaning: (952074, 14)
After cleaning: (928224, 14)


In [9]:
na_count = clean_df.isna().sum()
print(f"Number of rows with NA in 'station_distances': {na_count}")

Number of rows with NA in 'station_distances': OBSERVATION COUNT                0
COMMON NAME                      0
SCIENTIFIC NAME                  0
Order                        40480
STATE CODE                       0
COUNTY                           0
LOCALITY TYPE                    0
LATITUDE                         0
LONGITUDE                        0
TIME OBSERVATIONS STARTED        0
DURATION MINUTES             46723
i95_distance                     0
station_distances                0
assigned_station                 0
year_record                      0
month_record                     0
day_record                       0
hour_started                     0
day_of_week                      0
is_weekend                       0
is_migration                     0
extreme_weather                  0
daily_avg_noise                  0
peak_hour_noise                  0
overnight_noise                 28
rush_hour_noise                  0
total_daily_volume               0
8am_nois

Replacing null NUMBER OBSERVERS and null DURATION MINUTES with 1. <br>
<br>
Reason:  These values will be used to augment target.  It is reasonable to assume that for each observation, there were at least 1 observer for at least 1 min of observation occured for each observation entry submitted.Also, since we will be using these values to augment the target by division. The replacement value 1 would have the least impact on the resulting value.

In [ ]:
clean_df['NUMBER OBSERVERS'] = clean_df['NUMBER OBSERVERS'].fillna(1)
clean_df['DURATION MINUTES'] = clean_df['DURATION MINUTES'].fillna(1)

In [ ]:
# Birds per observer per hour
clean_df['obs_rate'] = clean_df['OBSERVATION COUNT'] / (clean_df['DURATION MINUTES'] / 60 * clean_df['NUMBER OBSERVERS'])

### 2.  Noise change 
- Use mean(daily-average_noise) as avg_noise
- Use row noise value - avg noise as the noise change

In [ ]:
#average 8 am noise
noise_mean = clean_df['8am_noise'].mean()
noise_mean

In [ ]:
clean_df['8am_noise_change'] = clean_df['8am_noise'] - noise_mean
clean_df.shape

### Target Transformation - BoxCox method

In [ ]:
### Observation_Count transformation ###
pt = PowerTransformer(method='box-cox', standardize=True)
y_train_boxcox = pt.fit_transform(y_train.values.reshape(-1, 1)).flatten()

print(f"\nSklearn lambda: {pt.lambdas_[0]:.4f}")
print("Sklearn Box-Cox + standardized statistics:")
print(f"Mean: {y_train_boxcox.mean():.2f}")
print(f"Std: {y_train_boxcox.std():.2f}")
print(f"Skewness: {stats.skew(y_train_boxcox):.2f}")

In [ ]:
# Transform validation and test sets using the already fitted transformer
y_val_boxcox = pt.transform(y_val.values.reshape(-1, 1)).flatten()
y_test_boxcox = pt.transform(y_test.values.reshape(-1, 1)).flatten()

In [ ]:
# OBSERVATION COUNT
fig, axes = plt.subplots(1, 2, figsize=(10,4))
# Original data
axes[0].hist(y_train, bins=50, alpha=0.7, color='red', density=True)
axes[0].set_title(f'Original Data (Skewness: {stats.skew(y_train.values):.2f})')
axes[0].set_xlabel('Values')
axes[0].set_ylabel('Density')

# Box-Cox transformed
axes[1].hist(y_train_boxcox, bins=50, alpha=0.7, color='blue', density=True)
axes[1].set_title(f'Box-Cox Transformed (Skewness: {stats.skew(y_train_boxcox):.2f})')
axes[1].set_xlabel('Transformed Values')
axes[1].set_ylabel('Density')

In [ ]:
# Inverse transformation function - for interference phase
def inverse_boxcox(transformed_data, lambda_val):
    """Inverse Box-Cox transformation"""
    if lambda_val == 0:
        return np.exp(transformed_data)
    else:
        return (transformed_data * lambda_val + 1)**(1/lambda_val)


### Aggregation
"A 10dB noise increase reduces daily station totals by 15 birds"

In [ ]:
# Aggregate by station and date
daily_station = df.groupby(['assigned_station', 'OBSERVATION DATE']).agg({
    'OBSERVATION COUNT': 'sum',  # Total birds seen that day
    'DURATION MINUTES': 'sum',   # Total observation effort
    'NUMBER OBSERVERS': 'sum',   # Total observers that day
    'daily_avg_noise': 'mean',   # Average noise for that day
    'extreme_weather': lambda x: x.mode()[0],  # Most common weather
    # Add other features as needed
}).reset_index()